<a href="https://colab.research.google.com/github/kekemelmohatle/SME-Legal-Assistant-SA/blob/main/SME_Legal_Assistant_SA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
import os

drive.mount("/content/drive")

PROJECT_PATH = "/content/drive/MyDrive/SME_Legal_Assistant_SA"

folders = ["data","models", "src", "docs"]
for folder in folders:
      os.makedirs(os.path.join(PROJECT_PATH, folder), exist_ok=True)


print("project structure created in google drive!")

In [ ]:
!pip install -q sentence-transformers scikit-learn

import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

categories = {
      "SARS_TAX": "VAT, income tax, payroll, dividends, corporate tax, SARS filing",
      "LABOR_LAW": "CCMA,, employee dismissal, leave policy, employment contract, labor dispute"

}

def identify_department(user_text):
  query_vec = model.encode([user_text])


  for dept, description in categories.items():
      dept_vec = model.encode([description])
      score = cosine_similarity(query_vec, dept_vec) [0][0]
      if score > 0.4:
          return f"This is a {dept} issue."


  return "I am not sure. Please ask about Tax or Labor Law."

print(identify_department(" How do i register my new staff for UIF?"))

In [ ]:
!pip install -q langchain langchain-community pypdf sentence-transformers chromadb

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

base_path = "/content/drive/MyDrive/SME_Legal_Assistant_SA/data/"
file_names = [
    "Legal-Pub-Guide-Gen09-Tax-Guide-for-Small-Businesses.pdf",
    "SARS-MultilingualTerminology-list-2022.pdf"
]

all_documents = []

for file_name in file_names:
    full_path = base_path + file_name
    print(f"Loading from: {full_path}")
    loader = PyPDFLoader(full_path)
    all_documents.extend(loader.load())

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
chunks = text_splitter.split_documents(all_documents)


embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="/content/drive/MyDrive/SME_Legal_Assistant_SA/models/vector_db"
)


print(f"\n SUCCESS: {len(chunks)} chunks processed and saved to your Google Drive!")

In [ ]:
def as_the_law(query):

  docs = vector_db.similarity_search(query, k=3)

  context = "\n--\n".join([d.page_content for d in docs])
  return f"Based on the SARS/CCMA docs, here is the relevant context:\n\n{context}"

print(as_the_law("What are the requirements for VAT registration for a small business"))

In [ ]:
!pip install -q transformers

In [ ]:
from transformers import NllbTokenizer, AutoModelForSeq2SeqLM

print("Initializing Translation Engine (NLLB-200)...")

# Load model and tokenizer once
tokenizer = NllbTokenizer.from_pretrained("facebook/nllb-200-distilled-600m")
model = AutoModelForSeq2SeqLM.from_pretrained("facebook/nllb-200-distilled-600m")

SA_LANGUAGES = {
    "English": "eng_Latn",
    "Sesotho": "sot_Latn",
    "Setswana": "tsn_Latn", # Corrected Setswana language code for NLLB
    "isiZulu": "zul_Latn",
    "isiXhosa": "xho_Latn",
    "afrikaans": "afr_Latn",
    "Sepedi": "nso_Latn",
    "isiSwati": "ssw_Latn",
    "Tshivenda": "ven_Latn",
    "isiTsonga": "tso_Latn",

}

DISCLAIMER_EN = "\n\n DISCLAIMER: This is an AI assistant, not a legal professional. Verify all infor with SARS/CCMA."

def ask_sme_assistant(user_query, target_language="isiZulu"):
  search_results = vector_db.similarity_search(user_query, k=2)
  english_context = "".join([doc.page_content for doc in search_results])

  lang_code = SA_LANGUAGES.get(target_language, "zul_Latn")

  # Use the loaded tokenizer and model for translation
  tokenizer.src_lang = "eng_Latn" # Set source language for the tokenizer
  inputs = tokenizer(english_context[:500], return_tensors="pt")

  # Generate translation, forcing the beginning-of-sequence token for the target language
  translated_tokens = model.generate(
      **inputs, forced_bos_token_id=tokenizer.convert_tokens_to_ids(f"__{lang_code}__"), max_length=512
  )

  translated_text = tokenizer.decode(translated_tokens[0], skip_special_tokens=True)

  print(f"\n--- ANSWER IN {target_language.upper()} ---")
  print(translated_text)
  print(DISCLAIMER_EN)


ask_sme_assistant("What are the tax rules for a small business?", target_language="Tshivenda")